# Whisper Medium Training - Sundanese ASR

Fine-tuning Whisper Medium on OpenSLR36 Sundanese dataset.

**Requirements:**
- GPU P100 (16GB VRAM)
- Dataset: `openslr36-sundanese-asr-prepared`

## Cell 1: Install Dependencies

Run this cell, then **Restart Session** before continuing.

In [ ]:
!pip uninstall -y datasets
!pip install datasets==2.21.0 soundfile librosa transformers evaluate jiwer
print("\n" + "="*50)
print("DONE! Now click: Runtime -> Restart Session")
print("Then run Cell 2")
print("="*50)

## Cell 2: Configuration

In [ ]:
import os
import sys
import json
import torch
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, List

# Kaggle paths
KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")

@dataclass
class Config:
    # Model
    model_name: str = "openai/whisper-medium"
    language: str = "sun"
    task: str = "transcribe"
    
    # Output
    output_dir: Path = KAGGLE_WORKING / "whisper-medium-sundanese"
    
    # Training
    max_steps: int = 5000
    batch_size: int = 8
    eval_batch_size: int = 4
    gradient_accumulation: int = 2
    learning_rate: float = 1e-5
    warmup_steps: int = 500
    
    # Checkpointing
    save_steps: int = 500
    eval_steps: int = 500
    logging_steps: int = 50
    
    # Performance
    fp16: bool = True
    gradient_checkpointing: bool = True

config = Config()
print(f"Model: {config.model_name}")
print(f"Steps: {config.max_steps}")
print(f"Batch: {config.batch_size} x {config.gradient_accumulation} = {config.batch_size * config.gradient_accumulation}")

## Cell 3: Helper Functions

In [ ]:
import os

def find_dataset():
    """Find dataset in Kaggle input."""
    search_paths = [
        KAGGLE_INPUT / "openslr36-sundanese-asr-prepared" / "kaggle_dataset" / "manifests",
        KAGGLE_INPUT / "openslr36-sundanese-asr-prepared" / "manifests",
        KAGGLE_INPUT / "openslr36-sundanese-asr-prepared",
    ]
    
    for path in search_paths:
        if (path / "train.json").exists():
            print(f"Found dataset: {path}")
            return path
    
    # Fallback: scan all
    for d in KAGGLE_INPUT.iterdir():
        if d.is_dir():
            for sub in ["kaggle_dataset/manifests", "manifests", ""]:
                check = d / sub if sub else d
                if (check / "train.json").exists():
                    print(f"Found dataset: {check}")
                    return check
    
    raise FileNotFoundError("Dataset not found!")


def fix_audio_path(old_path, audio_base):
    """Fix audio path to match actual Kaggle structure."""
    # Extract relative path after openslr36-audio
    if "openslr36-audio/" in old_path:
        rel = old_path.split("openslr36-audio/")[-1]
    elif "openslr36-audio\\" in old_path:
        rel = old_path.split("openslr36-audio\\")[-1]
    else:
        # Already relative or different format
        rel = old_path.split("/audio/")[-1] if "/audio/" in old_path else old_path
    
    rel = rel.replace("\\", "/")
    return f"{audio_base}/{rel}"


def load_manifest(path: Path, audio_base: Path = None, validate_audio: bool = True):
    """Load dataset from manifest JSON, optionally filtering missing audio."""
    from datasets import Dataset, Audio
    
    print(f"Loading {path.name}...")
    
    with open(path, 'r', encoding='utf-8') as f:
        entries = json.load(f)
    
    audio_base_str = str(audio_base) if audio_base else None
    
    # Process and validate audio paths
    valid_entries = []
    missing_count = 0
    
    for e in entries:
        ap = e['audio_path']
        
        # Fix the path
        if audio_base_str:
            fixed_path = fix_audio_path(ap, audio_base_str)
        else:
            fixed_path = ap.replace('\\', '/')
        
        # Validate if file exists
        if validate_audio and not os.path.exists(fixed_path):
            missing_count += 1
            continue
        
        valid_entries.append({
            'audio': fixed_path,
            'transcription': e['transcription']
        })
    
    if missing_count > 0:
        print(f"  ⚠ Skipped {missing_count:,} samples with missing audio files")
    
    if len(valid_entries) == 0:
        raise ValueError("No valid audio files found! Check dataset upload.")
    
    dataset = Dataset.from_dict({
        'audio': [e['audio'] for e in valid_entries],
        'transcription': [e['transcription'] for e in valid_entries],
    })
    dataset = dataset.cast_column('audio', Audio(sampling_rate=16000))
    
    print(f"  ✓ Loaded {len(dataset):,} valid samples")
    return dataset


def prepare_batch(batch, processor):
    """Prepare batch for training."""
    audio = batch["audio"]
    batch["input_features"] = processor.feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    batch["labels"] = processor.tokenizer(batch["transcription"]).input_ids
    return batch


@dataclass
class DataCollator:
    processor: Any
    
    def __call__(self, features: List[Dict]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]
        
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        
        batch["labels"] = labels
        return batch


def compute_wer(pred, processor, metric):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    
    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    
    return {"wer": 100 * metric.compute(predictions=pred_str, references=label_str)}

print("Helper functions loaded!")

## Cell 4: Check GPU & Load Dataset

In [ ]:
# Check GPU
print("="*60)
print("GPU Check")
print("="*60)

if not torch.cuda.is_available():
    print("ERROR: No GPU detected!")
    print("Go to: Settings -> Accelerator -> GPU P100")
else:
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Find dataset
print("\n" + "="*60)
print("Dataset")
print("="*60)

dataset_dir = find_dataset()
audio_base = dataset_dir.parent / "audio" if dataset_dir.name == "manifests" else dataset_dir / "audio"

if audio_base.exists():
    print(f"Audio base: {audio_base}")
    # Show available audio folders
    print("\nAvailable audio folders:")
    folders = sorted([f for f in audio_base.iterdir() if f.is_dir()])
    for folder in folders:
        file_count = sum(1 for _ in folder.rglob("*.flac"))
        print(f"  {folder.name}: {file_count:,} files")
    print(f"\nTotal folders: {len(folders)}")
else:
    print(f"Warning: Audio not found at {audio_base}")
    audio_base = None

## Cell 5: Load Model & Processor

In [ ]:
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
import evaluate

print(f"Loading {config.model_name}...")

processor = WhisperProcessor.from_pretrained(
    config.model_name,
    language=config.language,
    task=config.task
)

model = WhisperForConditionalGeneration.from_pretrained(config.model_name)
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
model.config.use_cache = False

if config.gradient_checkpointing:
    model.gradient_checkpointing_enable()

print(f"Model loaded! Parameters: {model.num_parameters():,}")

## Cell 6: Load & Prepare Datasets

In [ ]:
print("Loading datasets (filtering missing audio files)...")
print("This may take a few minutes to validate all paths...\n")

train_data = load_manifest(dataset_dir / "train.json", audio_base, validate_audio=True)
eval_data = load_manifest(dataset_dir / "validation.json", audio_base, validate_audio=True)

print("\n" + "="*60)
print("Preparing datasets (this takes ~30-60 min)...")
print("="*60)

train_data = train_data.map(
    lambda x: prepare_batch(x, processor),
    remove_columns=train_data.column_names,
    num_proc=2,
    desc="Preparing train",
)
eval_data = eval_data.map(
    lambda x: prepare_batch(x, processor),
    remove_columns=eval_data.column_names,
    num_proc=2,
    desc="Preparing eval",
)

print(f"\n✓ Train: {len(train_data):,} samples ready")
print(f"✓ Eval: {len(eval_data):,} samples ready")

## Cell 7: Setup Trainer

In [ ]:
config.output_dir.mkdir(parents=True, exist_ok=True)

data_collator = DataCollator(processor=processor)
metric = evaluate.load("wer")

training_args = Seq2SeqTrainingArguments(
    output_dir=str(config.output_dir),
    max_steps=config.max_steps,
    per_device_train_batch_size=config.batch_size,
    per_device_eval_batch_size=config.eval_batch_size,
    gradient_accumulation_steps=config.gradient_accumulation,
    learning_rate=config.learning_rate,
    warmup_steps=config.warmup_steps,
    fp16=config.fp16,
    eval_strategy="steps",
    eval_steps=config.eval_steps,
    save_strategy="steps",
    save_steps=config.save_steps,
    logging_steps=config.logging_steps,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    predict_with_generate=True,
    generation_max_length=225,
    dataloader_num_workers=2,
    report_to=["tensorboard"],
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=eval_data,
    data_collator=data_collator,
    compute_metrics=lambda pred: compute_wer(pred, processor, metric),
    processing_class=processor.feature_extractor,
)

print("Trainer ready!")
print(f"Output: {config.output_dir}")

## Cell 8: Train!

In [ ]:
print("="*60)
print("Starting Training")
print("="*60)
print(f"Model: {config.model_name}")
print(f"Batch: {config.batch_size} x {config.gradient_accumulation} = {config.batch_size * config.gradient_accumulation}")
print(f"Steps: {config.max_steps}")
print(f"LR: {config.learning_rate}")
print("="*60 + "\n")

trainer.train()

## Cell 9: Save Model

In [ ]:
print("Saving model...")
trainer.save_model()
processor.save_pretrained(config.output_dir)

print(f"\nModel saved to: {config.output_dir}")
print("\nTo download:")
print("1. Click 'Save Version' (top right)")
print("2. Select 'Save & Run All'")
print("3. After completion, go to Output tab")

## Cell 10: Evaluate on Test Set (Optional)

In [ ]:
test_file = dataset_dir / "test.json"

if test_file.exists():
    print("Evaluating on test set...")
    test_data = load_manifest(test_file, audio_base)
    test_data = test_data.map(
        lambda x: prepare_batch(x, processor),
        remove_columns=test_data.column_names,
        num_proc=2,
    )
    
    results = trainer.evaluate(test_data)
    print(f"\nTest WER: {results['eval_wer']:.2f}%")
    
    with open(config.output_dir / "test_results.json", 'w') as f:
        json.dump(results, f, indent=2)
else:
    print("No test.json found, skipping test evaluation.")